In [2]:
!pip install langchain langchain-google-genai langchain faiss-cpu langchain-text-splitters langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

In [5]:
os.environ["GEMINI_API_KEY"] = "ADD_YOUR_KEY"

In [7]:
text ="""ADD_KNOWLEDGE_BASE"""

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap =50)

In [9]:
docs = splitter.create_documents([text])

In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_5094/3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
vectorstore = FAISS.from_documents(docs,embeddings)

In [12]:
retrieve = vectorstore.as_retriever(search_kwargs = {"k" : 3})

In [13]:
def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs)

In [17]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""

    Use only the context below to answer the question.

    context:
    {context}

    question:
    {question}

    Answer:
    """
)

In [18]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [21]:
def rag_answer(query):
  r_docs = retrieve.invoke(query)
  context = format_docs(r_docs)
  full_prompt = prompt.format(context = context, question = query)
  response = llm.invoke(full_prompt)

  return response.content

In [24]:
query = "What is python used for?"
answer =rag_answer(query)

In [25]:
print("question: ", query)
print("\n")
print("answer", answer)

question:  What is python used for?


answer Python is used for making development easier and helping with a wide range of tasks, due to its huge collection of packages and standard libraries.
